In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [20]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDs


In [21]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

initialise our model here

In [22]:
MODEL = 'gpt-4.1-mini'

In [23]:
def pick_agent(user_message, history):
    messages = [
        {
                "role": "system",
                "content": "You are a routing classifier for feature requests."
                           "Decide which agent should answer: 'leave', 'devices' or 'general."

                           "Submission of requests will always be a specific agent, not general"
                           "Leave: Involves querying leave balance, drafting and submitting leave request such as medical, annual and family"
                           "Devices: Involves requesting for new device accessories such as cables and peripherals"
                           "General: Other unrelated queries"
        },
    ]

    if history:
        messages.extend(history)
    
    messages.append({"role": "user", "content": user_message})

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    intent = response.choices[0].message.content.lower()
    if "leave" in intent:
        return "leave"
    if "devices" in intent:
        return "devices"
    return "general"


In [24]:
from src.tools.common_tools import get_available_agents
from src.tools.leave_tools import (
    get_leave_balance,
    draft_leave_request,
    submit_leave_request,
    check_leave_status,
)
from src.tools.device_tools import get_available_devices, draft_device_request, submit_device_request


available_functions = {
    "get_leave_balance": get_leave_balance,
    "submit_leave_request": submit_leave_request,
    "draft_leave_request": draft_leave_request,
    "check_leave_status": check_leave_status,
    "get_available_devices": get_available_devices,
    "submit_device_request": submit_device_request,
    "draft_device_request": draft_device_request,
    "get_available_agents": get_available_agents,
}

3. The Agent Logic (The Loop)
I have removed stream=True for this example because handling streaming while doing function calling creates very complex code. It is safer to start with standard generation.

In [25]:
import json


def run_agent(config, message, history):
    # 1. Format history for OpenAI (cleaning up Gradio stuff)
    history_formatted = [{"role": h["role"], "content": h["content"]} for h in history]
    
    # 2. Add System Prompt
    system_prompt = config["system"]
    messages = [{"role": "system", "content": system_prompt}] + history_formatted + [{"role": "user", "content": message}]

    # --- FIRST API CALL (The "Thinking" Phase) ---
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=config["tools"],
        tool_choice="auto", 
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    # --- CHECK: Did the AI want to run a tool? ---
    if tool_calls:
        # A. Add the AI's "thought" (request to call tool) to the history
        messages.append(response_message) 
        
        # B. Iterate over requested tools (AI might call 2 at once!)
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f"--- SYSTEM:  Agent executed {function_name} tool")
            
            # C. Execute the actual Python function
            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)
            
            # D. Append the RESULT to the history
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )
            
        # --- SECOND API CALL (The "Answering" Phase) ---
        # Now the AI sees the tool output and generates the final text
        second_response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
        )
        return second_response.choices[0].message.content

    else:
        # If no tool was needed, just return the text
        return response_message.content

In [26]:
from src.agents.common_agent import agent_common
from src.agents.device_agent import agent_devices
from src.agents.leave_agent import agent_leaves

def unified_chat(message, history):
    agent_name = pick_agent(message, history)
    print(f"--- SYSTEM: Request routed to {agent_name} agent")

    if agent_name == "leave":
        config = agent_leaves()
    elif agent_name == "devices":
        config = agent_devices()
    else:
        config = agent_common()

    return run_agent(config, message, history)

In [ ]:
import gradio as gr

view = gr.ChatInterface(
    fn=unified_chat,
    title="SimpliAsk HR Agent",
    description="Check your terminal to see the Agent Traces (routing + tools) in real-time.",
    examples=[
        "I would like to apply for 7 days of annual leave.",
        "I would like to request for HDMI Cable.",
        "List workflows you can assist me with.",
    ],
)

view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://450f94def8f366eef0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


--- SYSTEM: Request routed to leave agent
--- SYSTEM: Request routed to leave agent
--- SYSTEM:  Agent executed draft_leave_request tool
--- SYSTEM: Draft created for mark_tan (annual leave) from 2023-12-06 to 2023-12-07 ---
--- SYSTEM: Request routed to leave agent
--- SYSTEM:  Agent executed draft_leave_request tool
--- SYSTEM: Draft created for mark_tan (annual leave) from 2023-12-06 to 2023-12-07 ---
--- SYSTEM: Request routed to leave agent
--- SYSTEM:  Agent executed draft_leave_request tool
--- SYSTEM: Draft created for mark_tan (annual leave) from 2023-12-06 to 2023-12-07 ---
--- SYSTEM: Request routed to leave agent
--- SYSTEM:  Agent executed submit_leave_request tool
--- SYSTEM: Submitting 2 days of annual leave for mark_tan with request ID MW88216 (status: pending) ---
--- SYSTEM: Request routed to leave agent
--- SYSTEM:  Agent executed check_leave_status tool
--- SYSTEM: Request routed to leave agent
